In [0]:
import hashlib
from pathlib import Path
from pyspark.sql import functions as F

account_number = "TEST00000001"

matches = (
    spark.table("workspace.migration_dev.document_registry")
    .filter(
        (F.col("account_number") == account_number)
        & (F.col("document_type") == "ACCOUNT_STATEMENT")
        & (F.col("statement_date") == F.lit("2026-09-30").cast("date"))
        & (F.col("batch_id") == "batch_001")
    )
    .collect()
)

if len(matches) != 1:
    raise RuntimeError(f"Expected one document; found {len(matches)}")

document = matches[0]
archive_path = Path(document["archive_path"])

# Restrict retrieval to the expected archive folder.
allowed_folder = Path(
    "/Volumes/workspace/migration_dev/archive/batch_001"
).resolve()

if archive_path.resolve().parent != allowed_folder:
    raise RuntimeError("Document path is outside the allowed archive folder")

content = archive_path.read_bytes()
actual_hash = hashlib.sha256(content).hexdigest()

if actual_hash != document["document_sha256"]:
    raise RuntimeError("Archive checksum mismatch; retrieval blocked")

print(f"Document ID: {document['document_id']}")
print(f"Account: {document['account_number']}")
print(f"Archive path: {archive_path}")
print(f"Retrieved bytes: {len(content)}")
print("PASS: Retrieved document matches the registry checksum")

Document ID: DOC000001
Account: TEST00000001
Archive path: /Volumes/workspace/migration_dev/archive/batch_001/DOC000001.pdf
Retrieved bytes: 1542
PASS: Retrieved document matches the registry checksum
